In [ ]:
# 1.1
import pandas as pd

df = pd.read_csv("week10_students.csv")
print("Shape:", df.shape)

corr = df[["quiz_avg", "assignments_avg", "attendance_pct"]].corrwith(df["score"])
print(corr)

Shape: (150, 9)
quiz_avg           0.800726
assignments_avg    0.535983
attendance_pct     0.238118
dtype: float64


In [ ]:
# 1.2
from sklearn.model_selection import train_test_split

X = df[["quiz_avg", "assignments_avg", "attendance_pct"]]
y = df["score"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

print("Train size:", len(X_train))
print("Test size :", len(X_test))

Train size: 112
Test size : 38


In [ ]:
# 2.1
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

for name, coef in zip(X.columns, model.coef_):
    print(f"{name:20s}: {coef: .4f}")
print(f"{'intercept':20s}: {model.intercept_: .4f}")

quiz_avg            :  0.5790
assignments_avg     :  0.3648
attendance_pct      :  0.1134
intercept           : -4.4282


In [ ]:
# 2.2
from sklearn.metrics import mean_absolute_error, r2_score

pred = model.predict(X_test)
mae = mean_absolute_error(y_test, pred)
r2  = r2_score(y_test, pred)

print(f"MAE : {mae:.4f}")
print(f"R²  : {r2:.4f}")

MAE : 3.3292
R²  : 0.8866


In [ ]:
# 2.3
new_student = pd.DataFrame(
    [[70, 75, 90]],
    columns=X.columns
)
print("New student prediction:", model.predict(new_student)[0])


me = pd.DataFrame([[45, 92, 89]], columns=X.columns)
print("My predicted score:", model.predict(me)[0])

New student prediction: 73.66410988729878
My predicted score: 65.27740052447764


In [ ]:
# 3.1
df["passed"] = (df["score"] >= 50).astype(int)
print(df["passed"].value_counts())

Xc = df[["quiz_avg", "assignments_avg", "attendance_pct"]]
yc = df["passed"]

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    Xc, yc, test_size=0.25, random_state=42
)

from sklearn.linear_model import LogisticRegression
clf = LogisticRegression(max_iter=1000)
clf.fit(Xc_train, yc_train)

passed
1    131
0     19
Name: count, dtype: int64


LogisticRegression(max_iter=1000)

In [ ]:
# 3.2
from sklearn.metrics import accuracy_score, confusion_matrix

pred_clf = clf.predict(Xc_test)

acc = accuracy_score(yc_test, pred_clf)
baseline = yc_test.mean()          # "everyone passes" accuracy
cm = confusion_matrix(yc_test, pred_clf)

print(f"Model accuracy   : {acc:.4f}")       # ~0.895
print(f"Lazy baseline    : {baseline:.4f}")  # ~0.895
print("Confusion matrix :\n", cm)

Model accuracy   : 0.8947
Lazy baseline    : 0.8947
Confusion matrix :
 [[ 3  1]
 [ 3 31]]


In [ ]:
# 4.1
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# quiz_avg only
X1 = df[["quiz_avg"]]
X1_train, X1_test, y1_train, y1_test = train_test_split(X1, y, test_size=0.25, random_state=42)
m1 = LinearRegression().fit(X1_train, y1_train)
r2_1 = r2_score(y1_test, m1.predict(X1_test))

# all three numeric features
X2 = df[["quiz_avg", "assignments_avg", "attendance_pct"]]
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y, test_size=0.25, random_state=42)
m2 = LinearRegression().fit(X2_train, y2_train)
r2_2 = r2_score(y2_test, m2.predict(X2_test))

# three numeric + program one-hot
X3 = pd.get_dummies(df[["quiz_avg", "assignments_avg", "attendance_pct", "program"]], columns=["program"])
X3_train, X3_test, y3_train, y3_test = train_test_split(X3, y, test_size=0.25, random_state=42)
m3 = LinearRegression().fit(X3_train, y3_train)
r2_3 = r2_score(y3_test, m3.predict(X3_test))

print(f"quiz_avg only          : R² = {r2_1:.4f}")
print(f"3 numeric features     : R² = {r2_2:.4f}")
print(f"3 numeric + program    : R² = {r2_3:.4f}")

quiz_avg only          : R² = 0.6242
3 numeric features     : R² = 0.8866
3 numeric + program    : R² = 0.8804


In [ ]:
# 5.1
import plotly.express as px

# Where the class stands
fig1 = px.histogram(df, x="score", nbins=12,
                    title="1. Where the class stands")
fig1.show()

# Who is at risk
fig2 = px.scatter(df, x="quiz_avg", y="score",
                  color="passed", hover_name="name",
                  title="2. Who is at risk")
fig2.show()

# Where to act
fig3 = px.bar(df.groupby("district", as_index=False)["score"].mean(),
              x="district", y="score",
              title="3. Where to act")
fig3.show()

In [ ]:
fig2.write_html("class_dashboard.html")
print("Saved class_dashboard.html")

Saved class_dashboard.html


In [ ]:
import os
print(os.path.exists("class_dashboard.html"))
print(os.path.getsize("class_dashboard.html"), "bytes")

True
4572277 bytes


In [ ]:
from google.colab import files
files.download("class_dashboard.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
fig_box = px.box(df, x="program", y="score", color="program",
                 title="Score distribution by program")
fig_box.show()

In [ ]:
## Part 6 — Model Card: Student Exam Score Predictor

1. Purpose
This model predicts a student's final exam score (0–100) from their quiz
average, assignment average, and attendance percentage. It is intended for
academic advisors and lecturers as an early-warning and planning tool — to
spot students who may need support before the exam, not to grade them.

2. Data & Features
Trained on 150 clean student records (no missing values) from
week10_students.csv.
- Features (X): quiz_avg, assignments_avg, attendance_pct
- Target (y): score
- Split: 75/25 train/test with random_state=42 → 112 train / 38 test students.

3. Performance (on unseen test data)
- MAE ≈ 3.3 points: on average, the predicted score is off by about 3 points.
  If the model says 70, the real score is typically between ~67 and ~73.
- R² ≈ 0.89: the model explains about 89% of the differences in students'
  scores. The remaining ~11% comes from factors we did not measure
  (study habits, home environment, exam-day conditions).

4. Risks & How to Use
- Who could be harmed: a student wrongly predicted to fail may face stigma,
  lowered expectations, or a self-fulfilling prophecy; a student wrongly
  predicted to pass may miss out on support they needed.
- How to use: as a triage flag only — to prompt a human advisor to check in
  with a student and to prioritise tutoring resources. Every flag must be
  paired with human judgement and the student's own input.
- How NOT to use: never for automatic pass/fail decisions, scholarship or
  disciplinary decisions, or sharing a prediction with a student without
  context and support.
- Bias note: the data may reflect historical inequities by district or
  gender; if used at scale, the model could automate those patterns.
  Subgroup error rates should be monitored before any real deployment.